## Bronze layer: EEG-Zahlungen Bewegungsdaten 2024.

Known data quirk in TransnetBW Teil2:
  ~318k rows had a phantom 11th column (header has trailing ';').
  ~3,622 of these had `_c10='1'` — a TransnetBW internal flag not
  in the documented schema.
  All affected rows have veraeusserungsform=5 (Sonstiges/bookkeeping)
  with eeg_zahlung=0, so they are filtered out in silver anyway.
  _rescued_data was cleared post-load via UPDATE since no real data
  was lost — only the phantom column.


In [0]:
"""Bronze layer: EEG-Zahlungen Bewegungsdaten 2024."""
from pyspark.sql import functions as F

LANDING_BASE = "/Volumes/eeg_dev/raw_files/landing/jahresabrechnung"
SCHEMA_BASE = "/Volumes/eeg_dev/raw_files/landing/_schemas/jahresabrechnung"

UENBS = ["50hertz", "amprion", "tennet", "transnetbw"]


def normalize_header(name: str) -> str:
    return (
        name.lower()
        .replace("ü", "ue").replace("ä", "ae")
        .replace("ö", "oe").replace("ß", "ss")
        .strip()
    )


def ingest(uenb: str) -> None:
    table = "eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten"
    schema_loc = f"{SCHEMA_BASE}/{uenb}"
    source_glob = f"{LANDING_BASE}/2024/{uenb}/*.csv"

    print(f"Ingesting {uenb} from {source_glob}")

    raw = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "false")
        .option("header", "true")
        .option("delimiter", ";")
        .option("encoding", "UTF-8")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(source_glob)
    )

    renamed_cols = [
        F.col(f"`{c}`").alias(normalize_header(c))
        for c in raw.columns if not c.startswith("_")
    ]
    rescued = [F.col(c) for c in raw.columns if c.startswith("_")]

    df = (
        raw.select(*renamed_cols, *rescued)
        .withColumn("_uenb", F.lit(uenb))
        .withColumn("_jahr", F.lit(2024))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", f"{schema_loc}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(table)
    )
    query.awaitTermination()
    print(f"  Done: {uenb}")


for uenb in UENBS:
    ingest(uenb)

print("\nAll ÜNB sources loaded into bronze.ntp_jahresabrechnung_bewegungsdaten")


In [0]:
%sql
-- Row count per ÜNB
SELECT _uenb, COUNT(*) AS n
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
GROUP BY _uenb ORDER BY _uenb;

-- Total should be ~8,179,452 (8,179,474 minus 22 header rows)

-- Verify schema is normalized
DESCRIBE eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten;

-- Spot check: any rescued data we should worry about?
SELECT COUNT(*) FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE _rescued_data IS NOT NULL;

-- Are all rescued rows from TransnetBW?
SELECT _uenb, COUNT(*) AS n
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE _rescued_data IS NOT NULL
GROUP BY _uenb;

-- And specifically Teil2?
SELECT _source_file, COUNT(*) AS n
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE _rescued_data IS NOT NULL
GROUP BY _source_file
ORDER BY n DESC;

SELECT _rescued_data, eeg_mastr_nr, eeg_zahlung
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE _rescued_data IS NOT NULL
LIMIT 5;

-- First confirm what's actually in there
SELECT _rescued_data, COUNT(*)
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE _rescued_data IS NOT NULL
GROUP BY _rescued_data
LIMIT 10;

-- Clear the rescue column for rows where it only contained the phantom column
UPDATE eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
SET _rescued_data = NULL
WHERE _rescued_data IS NOT NULL
  AND _rescued_data LIKE '%"_c10":""%';

  SELECT
  eeg_mastr_nr,
  eeg_anlagenschluessel,
  veraeusserungsform,
  verguetungskategorie,
  strommenge,
  eeg_zahlung,
  eeg_einnahmen,
  monat,
  _rescued_data
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE _rescued_data LIKE '%"_c10":"1"%'
LIMIT 5;

UPDATE eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
SET _rescued_data = NULL
WHERE _rescued_data LIKE '%"_c10":"1"%';

In [0]:
%sql
-- Total row count
SELECT COUNT(*) AS total
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten;
-- Expected: ~8,179,452;

-- By ÜNB
SELECT _uenb, COUNT(*) AS n
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
GROUP BY _uenb ORDER BY n DESC;

-- Sanity check on real money
SELECT
  _uenb,
  ROUND(SUM(CAST(REPLACE(eeg_zahlung, ',', '.') AS DOUBLE)) / 1e9, 2) AS milliarden_eur
FROM eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten
WHERE veraeusserungsform != '5'
GROUP BY _uenb
ORDER BY milliarden_eur DESC;